# Debug: Analyze Verbose Logs and Fix Intent Detection

In [ ]:
import os
from dotenv import load_dotenv
import json

load_dotenv(dotenv_path='../.env')
os.environ["OPENAI_API_KEY"] = os.environ['UNIFIED_LLM_KEY']
os.environ["OPENAI_API_BASE"] = os.environ['BASE_URL']

In [ ]:
from nemoguardrails import RailsConfig
from nemoguardrails.integrations.langchain.runnable_rails import RunnableRails

config_dir = "config/guardrails_intent"
config = RailsConfig.from_path(config_dir)
guardrails = RunnableRails(config=config, passthrough=False, verbose=True)

print("✅ Guardrails loaded with verbose=True")

## Test with Verbose Output

In [ ]:
test_input = "Find me books about Python programming"

print(f"\n{'='*70}")
print(f"Testing: {test_input}")
print(f"{'='*70}\n")

# This will show verbose logs
response = guardrails.invoke(
    {"input": test_input},
    config={"internal_events": True}
)

print(f"\n{'='*70}")
print("Analysis:")
print(f"{'='*70}")

## Analyze Internal Events

In [ ]:
# Check the response structure
print("\n1. Response Keys:")
print(f"   {list(response.keys())}")

# Check log
log = response.get("log", {})
print("\n2. Log Keys:")
print(f"   {list(log.keys())}")

# Get internal events
events = log.get("internal_events", [])
print(f"\n3. Total Internal Events: {len(events)}")

# Show all event types
print("\n4. Event Types:")
event_types = {}
for event in events:
    event_type = event.get("type", "unknown")
    event_types[event_type] = event_types.get(event_type, 0) + 1

for event_type, count in event_types.items():
    print(f"   {event_type}: {count}")

# Look for UserIntent events
print("\n5. UserIntent Events:")
user_intent_events = [e for e in events if e.get("type") == "UserIntent"]
if user_intent_events:
    print(f"   ✅ Found {len(user_intent_events)} UserIntent event(s)")
    for i, event in enumerate(user_intent_events):
        print(f"   Event {i+1}:")
        print(f"      Intent: {event.get('intent')}")
        print(f"      Full event: {json.dumps(event, indent=6)}")
else:
    print("   ❌ No UserIntent events found!")
    print("\n   This means intent detection is NOT working.")
    print("   Possible causes:")
    print("   - embeddings_only: true might not be supported")
    print("   - Embedding model not loading correctly")
    print("   - CoLang syntax error in rails.co")

## Show First 10 Events for Debugging

In [ ]:
print("\nFirst 10 Events (detailed):")
print("="*70)
for i, event in enumerate(events[:10]):
    print(f"\n{i+1}. Type: {event.get('type')}")
    print(f"   Event: {json.dumps(event, indent=3)[:200]}...")

## Test Improved Extract Intent Function

In [ ]:
def extract_intent_improved(response: dict, verbose: bool = True) -> str:
    """
    Improved intent extraction with detailed debugging.
    """
    try:
        # Get log
        log = response.get("log", {})
        if verbose:
            print(f"\n[DEBUG] Log exists: {bool(log)}")

        # Get events
        events = log.get("internal_events", [])
        if verbose:
            print(f"[DEBUG] Total events: {len(events)}")

        # Search for UserIntent events (iterate in reverse for most recent)
        for i, event in enumerate(reversed(events)):
            event_type = event.get("type")

            if event_type == "UserIntent":
                intent = event.get("intent", "general")
                if verbose:
                    print(f"[DEBUG] ✅ Found UserIntent at position {len(events) - i}")
                    print(f"[DEBUG] Intent value: '{intent}'")
                    print(f"[DEBUG] Full event: {event}")
                return intent

        # No UserIntent found
        if verbose:
            print(f"[DEBUG] ❌ No UserIntent events found")
            print(f"[DEBUG] Event types present: {set(e.get('type') for e in events)}")

        return "general"

    except Exception as e:
        if verbose:
            print(f"[DEBUG] ❌ Error: {e}")
            import traceback
            traceback.print_exc()
        return "general"

# Test it
print("\nTesting improved extract_intent function:")
print("="*70)
detected_intent = extract_intent_improved(response, verbose=True)
print(f"\nFinal Result: Intent = '{detected_intent}'")

## Diagnosis Summary

In [ ]:
print("\n" + "="*70)
print("DIAGNOSIS SUMMARY")
print("="*70)

has_user_intent = any(e.get("type") == "UserIntent" for e in events)

if has_user_intent:
    print("\n✅ Status: UserIntent events ARE being generated")
    print("   Issue: Extract function might be working correctly")
    print("   Next step: Check if intent matches expected value")
else:
    print("\n❌ Status: UserIntent events are NOT being generated")
    print("\n   ROOT CAUSE: Intent detection is not working at all")
    print("\n   SOLUTIONS:")
    print("   1. Change embeddings_only: false in config.yml")
    print("   2. Or use LLM-based intent detection instead")
    print("   3. Or upgrade NeMo Guardrails version")
    print("\n   Config to try:")
    print("   ```yaml")
    print("   rails:")
    print("     dialog:")
    print("       user_messages:")
    print("         embeddings_only: false  # Changed")
    print("   ```")